# Jeopardy! Data Explorer

Interactive SQL exploration of the scraped Jeopardy! data stored in DuckDB.
Uses [JupySQL](https://jupysql.ploomber.io/) `%%sql` magic for inline queries.

In [ ]:
import duckdb

conn = duckdb.connect('../data/jt3.duckdb')

%load_ext sql
%sql conn --alias duckdb

## Episodes & Contestants

In [ ]:
%%sql
SELECT *
FROM episodes
ORDER BY air_date DESC
LIMIT 10

In [ ]:
%%sql
SELECT e.game_id,
       e.show_number,
       e.air_date,
       c.name AS contestant,
       c.description
FROM episodes e
JOIN contestants c USING (game_id)
ORDER BY e.air_date DESC
LIMIT 20

## Clues & Categories

In [ ]:
%%sql
SELECT cat.name AS category,
       cl.value,
       cl.text AS clue,
       cl.correct_response,
       cl.is_daily_double
FROM clues cl
JOIN categories cat USING (game_id, round_index, category_index)
ORDER BY cl.game_id DESC, cl.round_index, cl.category_index
LIMIT 20

## Capture Results to Python

Use the `<<` syntax to store query results in a Python variable.

In [ ]:
%%sql category_counts <<
SELECT cat.name AS category,
       COUNT(*) AS clue_count
FROM categories cat
JOIN clues cl USING (game_id, round_index, category_index)
GROUP BY cat.name
ORDER BY clue_count DESC
LIMIT 25

In [ ]:
category_counts

In [ ]:
%%sql --save football_categories
SELECT DISTINCT cat.name AS category
FROM categories cat
WHERE cat.name LIKE '% NFL%'
    OR cat.name LIKE '%NFL %'
    OR cat.name LIKE '% FOOTBALL%'
    OR cat.name LIKE '%FOOTBALL %'

In [ ]:
%%sql --with football_categories
SELECT cat.name AS category,
        #  e.air_date,
         cl.value,
         cl.text AS clue,
         cl.correct_response,
        #  cl.is_daily_double
FROM clues cl
JOIN categories cat USING (game_id, round_index, category_index)
JOIN episodes e USING (game_id)
WHERE cat.name IN (SELECT category FROM football_categories)

In [ ]:
%%sql --with football_categories football_df <<
SELECT cat.name AS category,
        #  e.air_date,
         cl.value,
         cl.text AS clue,
         cl.correct_response,
        #  cl.is_daily_double
FROM clues cl
JOIN categories cat USING (game_id, round_index, category_index)
JOIN episodes e USING (game_id)
WHERE cat.name IN (SELECT category FROM football_categories)

In [ ]:
import polars as pl
football_df.PolarsDataFrame().write_csv('../data/football_clues.csv')